# Dynamic Gradient Flow Monitoring

## Install

```bash
pip install graphyco
```

---

## Setup & Bridge Initialization

Initialize DynamicExecutionBridge on a residual block.

In [ ]:
import sys, os
for p in [os.path.abspath("../../validation"), os.path.abspath("../validation"), os.path.abspath("validation")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
import sys, os
import torch
import torch.nn as nn
import pandas as pd
from graphyco.bridge.grad_bridge import DynamicExecutionBridge
from validate_framework import ResNetBlock

torch.manual_seed(42)
model = ResNetBlock(dim=64)
bridge = DynamicExecutionBridge(model, mode="trace")
print(f"Traced {len(bridge.static_state.nodes)} nodes and {len(bridge.static_state.edges)} edges.")

## Multi-Step Execution & Flow Tracking

Execute 5 forward/backward steps with loss computation.

In [ ]:
inputs = torch.randn(8, 64)
opt = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(5):
    batch = inputs + 0.05 * torch.randn_like(inputs)
    state = bridge.step(inputs=batch, optimizer=opt, loss_fn=lambda out: out.sum())
    bneck = state.dynamic_evaluation["bottleneck"]
    print(f"Step {state.step}: Bottleneck = '{bneck['bottleneck_node']}', G_max = {bneck['max_concentration']:.3f}")

## Node Records and Tensor Statistics

Inspect activation RMS, gradient RMS, and sensitivity ratio per node.

In [ ]:
latest = bridge.trajectory[-1]
rows = []
for nid, rec in sorted(latest.node_records.items()):
    act_rms = rec.activation.rms if rec.activation else 0.0
    grad_rms = rec.activation_gradient.rms if rec.activation_gradient else 0.0
    rows.append({
        "Node": nid,
        "Type": rec.node_type,
        "Has Params": rec.has_parameters,
        "Act RMS": f"{act_rms:.4f}",
        "Grad RMS": f"{grad_rms:.4f}",
        "Ratio (Grad/Act)": f"{rec.ratio_grad_act:.4f}" if rec.ratio_grad_act is not None else "N/A"
    })
pd.DataFrame(rows)

## Directed Edge Gradient Attenuation

Inspect backward attenuation ratio beta and logarithmic attenuation Delta along edges.

In [ ]:
attenuation = latest.dynamic_evaluation["attenuation"]
edge_rows = []
for edge_key, att in list(attenuation.items())[:6]:
    edge_rows.append({
        "Edge": edge_key,
        "Backward Ratio": f"{att['backward_ratio']:.4f}",
        "Log Ratio": f"{att['log_backward_ratio']:+.4f}",
        "Src Grad L2": f"{att['source_grad_l2']:.4f}",
        "Tgt Grad L2": f"{att['target_grad_l2']:.4f}"
    })
pd.DataFrame(edge_rows)

## Structural-Functional Classification (Types I - IV)

Categorize nodes into 4 quadrants based on topology vs. gradient magnitude.

In [ ]:
corr = latest.dynamic_evaluation["correspondence"]
print("Node Category Counts:")
for cat, count in corr["counts"].items():
    print(f"  {cat:<10}: {count}")

print()
print(f"Spearman rho(Topology, Gradient): {corr['spearman_topo_grad']:.4f} (p={corr['spearman_pvalue']:.4e})")

## Temporal Stability Analysis

Compute temporal coefficient of variation (CV = sigma / mu) across execution steps.

In [ ]:
temporal = latest.dynamic_evaluation.get("temporal", {})
t_rows = []
for nid, t_stat in sorted(temporal.items())[:6]:
    t_rows.append({
        "Node": nid,
        "Mean Grad": f"{t_stat['mean_gradient']:.4f}",
        "Std Grad": f"{t_stat['std_gradient']:.4f}",
        "CV": f"{t_stat['cv']:.4f}",
        "Classification": t_stat["classification"]
    })
pd.DataFrame(t_rows)

## Serialization to JSON and CSV

Export monitored trajectory to disk.

In [ ]:
json_path = "resnet_step5.json"
csv_path = "resnet_step5_nodes.csv"
latest.to_json(path=json_path)
latest.to_csv_nodes(path=csv_path)

print(f"Exported JSON size: {os.path.getsize(json_path):,} bytes")
print(f"Exported CSV size : {os.path.getsize(csv_path):,} bytes")

# Clean up exported files
if os.path.exists(json_path): os.remove(json_path)
if os.path.exists(csv_path): os.remove(csv_path)